# 🧠 Mental Health Google Form — Research Analysis (v2 — Fixed)
### Ensemble: Your Model (LR/SVM/RF) + BART Zero-Shot

---
> **Run cells in order. Only edit the `⚙️ CONFIGURATION` cell.**

**15 Target Classes:**  
`addiction` · `adhd` · `alcoholism` · `anxiety` · `autism` · `bipolarreddit` · `bpd` · 
`depression` · `healthanxiety` · `lonely` · `mentalhealth` · `ptsd` · `schizophrenia` · 
`socialanxiety` · `suicidewatch`

### 🔴 Three Root Causes Fixed in This Version
1. **ZIP file**: `Mentalhealth.csv` is actually a ZIP — auto-extracted here
2. **Dual column layout**: Rows 1-7 had text in cols 35/36, rows 8+ in cols 2/3 — now both are read
3. **Thresholds too high**: 0.38/0.28 are for Reddit posts; survey language needs 0.20/0.12


In [ ]:
from pathlib import Path as _ProjectPath
import sys as _project_sys

_project_root = next(
    (
        p
        for p in (_ProjectPath.cwd(), *_ProjectPath.cwd().parents)
        if (p / "project_config.json").is_file()
    ),
    None,
)
if _project_root is None:
    raise FileNotFoundError(
        "Open VS Code in the organized project root before running this notebook."
    )
if str(_project_root / "tools") not in _project_sys.path:
    _project_sys.path.insert(0, str(_project_root / "tools"))
from project_paths import ProjectPaths as _ProjectPaths

_paths = _ProjectPaths(_project_root, 'gg')

import os, warnings, re, zipfile

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import shap
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update(
    {
        'figure.dpi': 120,
        'figure.facecolor': 'white',
        'axes.facecolor': '#f9f9f9',
        'axes.grid': True,
        'grid.alpha': 0.3,
        'font.family': 'DejaVu Sans',
    }
)
CLASS_NAMES = [
    'addiction',
    'adhd',
    'alcoholism',
    'anxiety',
    'autism',
    'bipolarreddit',
    'bpd',
    'depression',
    'healthanxiety',
    'lonely',
    'mentalhealth',
    'ptsd',
    'schizophrenia',
    'socialanxiety',
    'suicidewatch',
]
PALETTE = [
    '#e74c3c',
    '#3498db',
    '#2ecc71',
    '#f39c12',
    '#9b59b6',
    '#1abc9c',
    '#e67e22',
    '#34495e',
    '#e91e63',
    '#00bcd4',
    '#8bc34a',
    '#ff5722',
    '#607d8b',
    '#ffc107',
    '#795548',
]
CLASS_COLOR = {cls: PALETTE[i] for i, cls in enumerate(CLASS_NAMES)}
print('✅ Libraries loaded')
print(f'   SHAP {shap.__version__} | Pandas {pd.__version__} | NumPy {np.__version__}')


In [ ]:
CSV_FILE = _paths.input_path('mental_health_response.csv')

Q1_COL_LAYOUT_A = None
Q2_COL_LAYOUT_A = None
Q1_COL_LAYOUT_B = None
Q2_COL_LAYOUT_B = None


CONFIDENCE_THRESHOLD = 0.35
BORDERLINE_LOWER = 0.25


REDDIT_WEIGHT = 0.35
BART_WEIGHT = 0.65


MODEL_DIR = _paths.classical_dir
TFIDF_FILE = 'tfidf_vectorizer.pkl'
LABEL_FILE = 'label_encoder.pkl'
LR_FILE = 'logistic_regression_model.pkl'
SVM_FILE = 'linear_svm_model.pkl'
RF_FILE = 'random_forest_model.pkl'

print('✅ Configuration set')
print(f'   CSV                : {CSV_FILE}')
print(f'   Condition threshold: {CONFIDENCE_THRESHOLD}')
print(f'   Borderline         : {BORDERLINE_LOWER}')
print(f'   Reddit / BART mix  : {REDDIT_WEIGHT:.0%} / {BART_WEIGHT:.0%}')


In [ ]:
print('🤖 Loading pre-trained models...')
print('=' * 70)

if not os.path.exists(os.path.join(MODEL_DIR, TFIDF_FILE)):
    MODEL_DIR = _paths.classical_dir

print(f'   📂 Loading from: {MODEL_DIR}')


def _load(filename):
    path = os.path.join(MODEL_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f'❌ Cannot find: {path}')
    return joblib.load(_paths.input_path(path))


tfidf = _load(TFIDF_FILE)
print(f'   ✅ TF-IDF ({tfidf.max_features} features)')
le = _load(LABEL_FILE)
print(f'   ✅ Label encoder ({len(le.classes_)} classes)')
lr = _load(LR_FILE)
print('   ✅ Logistic Regression')
svm = _load(SVM_FILE)
print('   ✅ Linear SVM')
rf = _load(RF_FILE)
print('   ✅ Random Forest')

models = {'Logistic Regression': lr, 'Linear SVM': svm, 'Random Forest': rf}
print(f'\n🎯 {len(le.classes_)} classes: {list(le.classes_)}')


In [ ]:
print('📂 Loading CSV (ZIP-aware)...')
print('=' * 70)


csv_path = _paths.input_path(
    os.path.join(MODEL_DIR, CSV_FILE) if not os.path.isabs(CSV_FILE) else CSV_FILE
)


def _resolve_csv(path):
    """If path is a ZIP, extract the first .csv inside and return its path."""
    import zipfile

    if zipfile.is_zipfile(path):
        print(f'   ℹ️  Detected ZIP archive: {os.path.basename(path)}')
        extract_dir = _paths.output_path(
            os.path.join(os.path.dirname(path), '_extracted_csv')
        )
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(_paths.input_path(path), 'r') as z:
            csv_names = [n for n in z.namelist() if n.lower().endswith('.csv')]
            if not csv_names:
                raise ValueError('❌ No .csv file found inside the ZIP.')
            z.extract(csv_names[0], extract_dir)
            extracted = os.path.join(extract_dir, csv_names[0])
            print(f'   ✅ Extracted: {csv_names[0]}')
            return extracted
    return path


csv_path = _paths.input_path(_resolve_csv(csv_path))


raw_df = None
for enc in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
    try:
        raw_df = pd.read_csv(
            _paths.input_path(csv_path),
            encoding=enc,
            on_bad_lines='skip',
            engine='python',
        )
        print(f'   ✅ Encoding "{enc}" worked')
        break
    except UnicodeDecodeError:
        print(f'   ⚠️  "{enc}" failed, trying next...')

if raw_df is None:
    raise ValueError('❌ Could not read CSV.')

print(f'\n✅ Loaded  →  {raw_df.shape[0]} rows × {raw_df.shape[1]} columns')
for i, col in enumerate(raw_df.columns):
    print(f'   [{i:2d}] {col[:90]}')
raw_df.head(3)


In [ ]:
print('🔍 Extracting text — universal layout...')
print('=' * 70)


def find_col_keyword(df, keyword, fallback_index=None):
    if fallback_index is not None:
        return df.columns[fallback_index]
    matches = [c for c in df.columns if keyword.lower() in c.lower()]
    return matches[0] if matches else None


col_a_q1 = find_col_keyword(raw_df, 'heaviest', Q1_COL_LAYOUT_A)
col_a_q2 = find_col_keyword(raw_df, 'alone', Q2_COL_LAYOUT_A)

col_b_rating = raw_df.columns[34] if raw_df.shape[1] > 34 else None
col_b_q1 = raw_df.columns[Q1_COL_LAYOUT_B or 35] if raw_df.shape[1] > 35 else None
col_b_q2 = raw_df.columns[Q2_COL_LAYOUT_B or 36] if raw_df.shape[1] > 36 else None
likert_cols = [raw_df.columns[j] for j in range(4, 34) if raw_df.shape[1] > j]

print(f'   Layout A Q1 : {str(col_a_q1)[:60] if col_a_q1 else "NOT FOUND"}')
print(f'   Layout A Q2 : {str(col_a_q2)[:60] if col_a_q2 else "NOT FOUND"}')
print(f'   Layout B    : cols {col_b_rating}, {col_b_q1}, {col_b_q2}')
print(f'   Likert cols : {len(likert_cols)} found')
print(f'   Total cols  : {raw_df.shape[1]}')


def likert_to_text(row, likert_cols):
    vals = [str(row[c]).strip() for c in likert_cols if pd.notna(row[c])]
    if not vals:
        return ''
    freq_map = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Very Often': 4}
    scores = [freq_map.get(v, 2) for v in vals]
    avg = sum(scores) / len(scores)
    often_count = sum(1 for v in vals if v in ('Often', 'Very Often'))
    if avg >= 3.0 or often_count >= 10:
        return (
            'I often feel overwhelmed, anxious or low. '
            'Many things bother me frequently. I struggle with my emotions often.'
        )
    elif avg >= 2.0 or often_count >= 5:
        return (
            'Sometimes I feel stressed or anxious. '
            'Some things bother me from time to time.'
        )
    else:
        return 'I mostly feel okay. Things are generally manageable for me.'


SKIP_COLS = {'Timestamp', 'Email address'}


def extract_row_text(row):
    parts = []

    for col in [col_a_q1, col_a_q2]:
        if col and pd.notna(row[col]):
            val = str(row[col]).strip()
            if len(val) > 10 and val.lower() not in ('nan', 'none', ''):
                parts.append(val)

    if col_b_rating and pd.notna(row[col_b_rating]):
        sr = str(row[col_b_rating]).strip()
        if sr not in ('nan', 'none', '') and len(sr) > 5:
            parts.append(f'Self-reported mood: {sr}.')
    for col in [col_b_q1, col_b_q2]:
        if col and pd.notna(row[col]):
            val = str(row[col]).strip()
            if len(val) > 10 and val.lower() not in ('nan', 'none', ''):
                parts.append(val)

    if not parts or sum(len(p) for p in parts) < 15:
        extra = []
        for col in raw_df.columns:

            if any(skip.lower() in col.lower() for skip in SKIP_COLS):
                continue

            if col in [col_a_q1, col_a_q2, col_b_q1, col_b_q2, col_b_rating]:
                continue
            val = str(row[col]).strip()

            if (
                len(val) > 15
                and val.lower() not in ('nan', 'none', '')
                and not val.replace(' ', '').isdigit()
                and val not in ('Never', 'Rarely', 'Sometimes', 'Often', 'Very Often')
            ):
                extra.append(val)

        extra.sort(key=len, reverse=True)
        parts.extend(extra[:2])

    if not parts and likert_cols:
        lt = likert_to_text(row, likert_cols)
        if lt:
            parts.append(lt)

    combined = ' '.join(parts).strip()
    return re.sub(r'\s+', ' ', combined)


df = pd.DataFrame()
df['text'] = raw_df.apply(extract_row_text, axis=1)
df['participant_id'] = [f'P{i+1:03d}' for i in range(len(df))]

ts_col = [c for c in raw_df.columns if 'timestamp' in c.lower()]
if ts_col:
    df['timestamp'] = pd.to_datetime(raw_df[ts_col[0]], errors='coerce')
    print(f'   ✅ Timestamp detected')


empty_count = (df['text'].str.len() <= 10).sum()
if empty_count > 0:
    print(f'\n   ⚠️  {empty_count} rows still empty after all fallbacks:')
    for i, row in df[df['text'].str.len() <= 10].iterrows():
        print(
            f'      Row {i}: raw cols with data = '
            f'{[c for c in raw_df.columns if pd.notna(raw_df.loc[i, c]) and len(str(raw_df.loc[i, c])) > 5][:5]}'
        )

df = df[df['text'].str.len() > 5].reset_index(drop=True)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
n = len(df)

print(f'\n✅ {n} / {len(raw_df)} participants extracted')
print(
    f'   Word count : min={df["word_count"].min()}, '
    f'max={df["word_count"].max()}, mean={df["word_count"].mean():.0f}'
)
print('\n📋 Sample:')
for _, row in df.head(4).iterrows():
    print(f'   {row["participant_id"]}: {row["text"][:100]}...')


In [ ]:
from transformers import pipeline
import torch, re
from collections import Counter

print('🤖 Loading zero-shot classifier...')
zs_classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=0 if torch.cuda.is_available() else -1,
)
print('   ✅ facebook/bart-large-mnli loaded')


VALID_CONDITIONS = [
    'addiction',
    'adhd',
    'alcoholism',
    'anxiety',
    'autism',
    'bipolarreddit',
    'bpd',
    'depression',
    'healthanxiety',
    'lonely',
    'ptsd',
    'schizophrenia',
    'socialanxiety',
    'suicidewatch',
]


ZS_LABELS = [
    'feeling persistently sad, empty, or hopeless with no energy',
    'excessive worry, overthinking, or constant anxiety and stress',
    'suicidal thoughts, wanting to die, or feeling life is not worth living',
    'attention difficulties, impulsivity, restlessness, or ADHD symptoms',
    'extreme mood swings between very high energy and very low crashes',
    'emotional instability, fear of abandonment, or identity confusion',
    'trauma, flashbacks, intrusive memories, or past abuse',
    'fear of social situations, embarrassment, or being judged by others',
    'excessive worry about physical health or body symptoms',
    'deep loneliness, social isolation, or feeling disconnected',
    'paranoia, hallucinations, or losing touch with reality',
    'alcohol misuse or dependency',
    'drug addiction or substance misuse',
    'difficulty understanding social cues or autism spectrum traits',
]
ZS_TO_CONDITION = {
    'feeling persistently sad, empty, or hopeless with no energy': 'depression',
    'excessive worry, overthinking, or constant anxiety and stress': 'anxiety',
    'suicidal thoughts, wanting to die, or feeling life is not worth living': 'suicidewatch',
    'attention difficulties, impulsivity, restlessness, or ADHD symptoms': 'adhd',
    'extreme mood swings between very high energy and very low crashes': 'bipolarreddit',
    'emotional instability, fear of abandonment, or identity confusion': 'bpd',
    'trauma, flashbacks, intrusive memories, or past abuse': 'ptsd',
    'fear of social situations, embarrassment, or being judged by others': 'socialanxiety',
    'excessive worry about physical health or body symptoms': 'healthanxiety',
    'deep loneliness, social isolation, or feeling disconnected': 'lonely',
    'paranoia, hallucinations, or losing touch with reality': 'schizophrenia',
    'alcohol misuse or dependency': 'alcoholism',
    'drug addiction or substance misuse': 'addiction',
    'difficulty understanding social cues or autism spectrum traits': 'autism',
}


REDDIT_FREQ = {
    'depression': 0.2848,
    'suicidewatch': 0.1606,
    'anxiety': 0.1400,
    'adhd': 0.1108,
    'bpd': 0.0590,
    'lonely': 0.0574,
    'socialanxiety': 0.0558,
    'autism': 0.0215,
    'schizophrenia': 0.0211,
    'healthanxiety': 0.0210,
    'ptsd': 0.0210,
    'addiction': 0.0185,
    'alcoholism': 0.0143,
    'bipolarreddit': 0.0140,
}

UNIFORM = 1.0 / len(VALID_CONDITIONS)
CORRECTION = np.array([UNIFORM / REDDIT_FREQ.get(c, UNIFORM) for c in le.classes_])


_valid_in_le = [c for c in le.classes_ if c in VALID_CONDITIONS]
if 'mentalhealth' in le.classes_:
    print(
        "   ⚠️  Label encoder pkl contains 'mentalhealth' — filtered via VALID_CONDITIONS."
    )
assert (
    len(_valid_in_le) == 14
), f"Expected 14 valid conditions in le, found {len(_valid_in_le)}: {_valid_in_le}"
print(
    f"   ✅ Safety check passed — {len(_valid_in_le)} valid output conditions confirmed"
)


CONFIDENCE_THRESHOLD = 0.35
BORDERLINE_LOWER = 0.25


KEYWORD_RULES = [
    (
        {
            'should i live',
            'want to die',
            'beauty of death',
            'loving death',
            'live or die',
            'end my life',
            'no reason to live',
            'suicide',
            'suicidal',
            'kill myself',
            'better off dead',
        },
        'suicidewatch',
        5.0,
    ),
    (
        {
            'flashback',
            'trauma',
            'intrusive',
            'haunting',
            'cant forget',
            'keeps coming back',
            'replaying',
            'past abuse',
            'triggered',
        },
        'ptsd',
        3.0,
    ),
    (
        {
            'which version of me',
            'wake up not knowing who i am',
            'euphoric then crash',
            'manic episode',
            'no sleep for days',
        },
        'bipolarreddit',
        1.5,
    ),
    (
        {
            'two people',
            'two versions',
            'dont know who i am',
            'fear of abandonment',
            'push and pull',
        },
        'bpd',
        2.0,
    ),
    (
        {
            'worthless',
            'hopeless',
            'empty inside',
            'no energy',
            'cant get out of bed',
            'useless',
            'numb',
            'void',
            'feel nothing',
        },
        'depression',
        2.0,
    ),
    (
        {
            'overthinking',
            'cant stop thinking',
            'racing mind',
            'constant worry',
            'always anxious',
            'panic',
        },
        'anxiety',
        1.5,
    ),
    (
        {
            'no one understands',
            'feel alone',
            'nobody cares',
            'completely alone',
            'disconnected',
            'no friends',
        },
        'lonely',
        2.0,
    ),
]


DISTRESS_STRONG = {
    'worthless',
    'hopeless',
    'suicidal',
    'kill myself',
    'want to die',
    'flashback',
    'trauma',
    'numb',
    'void',
    'empty inside',
    'end my life',
    'no reason to live',
    'better off dead',
    'abuse',
    'intrusive',
}


def run_your_model(texts):
    X = tfidf.transform(texts)
    probas = []
    for model in models.values():
        if hasattr(model, 'predict_proba'):
            p = model.predict_proba(X)
        else:
            d = model.decision_function(X)
            p = np.exp(d - d.max(axis=1, keepdims=True))
            p /= p.sum(axis=1, keepdims=True)
        probas.append(p)
    avg = np.stack(probas).mean(axis=0) * CORRECTION
    return avg / avg.sum(axis=1, keepdims=True)


def run_zeroshot(text):
    result = zs_classifier(text[:1024], candidate_labels=ZS_LABELS, multi_label=True)
    out = {c: 0.0 for c in VALID_CONDITIONS}
    for lbl, score in zip(result['labels'], result['scores']):
        cond = ZS_TO_CONDITION.get(lbl)
        if cond and cond in out:
            if score >= 0.20:
                out[cond] += float(score)
    total = sum(out.values()) or 1.0
    return {k: v / total for k, v in out.items()}


def apply_keyword_rules(blended, text):
    text_lower = text.lower()
    for keywords, condition, multiplier in KEYWORD_RULES:
        if any(kw in text_lower for kw in keywords):
            if condition in blended:
                blended[condition] *= multiplier
    total = sum(blended.values()) or 1.0
    return {k: v / total for k, v in blended.items()}


def classify_batch(texts, threshold=CONFIDENCE_THRESHOLD):
    your_proba = run_your_model(texts)
    rows, blended_rows = [], []
    print(f'   Classifying {len(texts)} participants...')

    for i, text in enumerate(texts):
        text_lower = text.lower()
        text_words = set(re.sub(r"[^a-z ]", "", text_lower).split())

        your_dict = {
            le.classes_[j]: float(your_proba[i][j]) for j in range(len(le.classes_))
        }

        your_valid = {c: your_dict.get(c, 0.0) for c in VALID_CONDITIONS}
        t = sum(your_valid.values()) or 1.0
        your_valid = {k: v / t for k, v in your_valid.items()}

        zs_dict = run_zeroshot(text)

        blended = {
            c: REDDIT_WEIGHT * your_valid.get(c, 0.0)
            + BART_WEIGHT * zs_dict.get(c, 0.0)
            for c in VALID_CONDITIONS
        }

        blended = apply_keyword_rules(blended, text)

        sorted_blended = sorted(blended.items(), key=lambda x: -x[1])
        final_label = sorted_blended[0][0]
        final_conf = sorted_blended[0][1]

        your_top = max(your_valid, key=your_valid.get)
        zs_top = max(zs_dict, key=zs_dict.get)
        agree = your_top == zs_top

        has_strong_distress = bool(text_words & DISTRESS_STRONG)

        if final_label == 'suicidewatch':
            risk_level, status = 'high', 'condition_detected'
        elif not has_strong_distress and final_conf < BORDERLINE_LOWER:
            final_label = 'normal'
            risk_level, status = 'low', 'normal'
        elif final_conf >= threshold:
            risk_level = 'high'
            status = 'condition_detected'
        elif final_conf >= BORDERLINE_LOWER:
            risk_level, status = 'borderline', 'borderline'
        else:
            final_label = 'normal'
            risk_level, status = 'low', 'normal'

        top5 = sorted_blended[:5]
        rows.append(
            {
                'participant_id': df['participant_id'].iloc[i],
                'classification': final_label,
                'confidence': round(final_conf, 4),
                'status': status,
                'risk_level': risk_level,
                'models_agree': agree,
                'your_model_top': your_top,
                'zeroshot_top': zs_top,
                'top1_condition': top5[0][0],
                'top1_prob': round(top5[0][1], 4),
                'top2_condition': top5[1][0] if len(top5) > 1 else '',
                'top2_prob': round(top5[1][1], 4) if len(top5) > 1 else 0.0,
                'word_count': df['word_count'].iloc[i],
            }
        )
        blended_full = {l: blended.get(l, 0.0) for l in VALID_CONDITIONS}
        blended_full['participant_id'] = df['participant_id'].iloc[i]
        blended_rows.append(blended_full)

        if (i + 1) % 5 == 0 or (i + 1) == len(texts):
            print(f'   ... {i+1}/{len(texts)} done')

    global blended_matrix_df
    blended_matrix_df = pd.DataFrame(blended_rows).set_index('participant_id')
    return pd.DataFrame(rows)


print('\n✅ Ensemble ready  —  mentalhealth removed from output')
print('   Valid output conditions:', ', '.join(VALID_CONDITIONS))


In [ ]:
print('🚀 Running Ensemble Classification...')
print('=' * 70)

texts_list = df['text'].tolist()
results_df = classify_batch(texts_list)


assert (
    'mentalhealth' not in results_df['classification'].values
), "mentalhealth still in output — check classify_batch"

n = len(results_df)
n_normal = (results_df['status'] == 'normal').sum()
n_borderline = (results_df['risk_level'] == 'borderline').sum()
n_condition = (results_df['risk_level'] == 'high').sum()
n_agree = results_df['models_agree'].sum()

print(f'\n✅ Done — {n} participants classified')
print(f'   🟢 Normal            : {n_normal}  ({n_normal/n*100:.1f}%)')
print(f'   🟡 Borderline        : {n_borderline}  ({n_borderline/n*100:.1f}%)')
print(f'   🔴 Condition detected: {n_condition}  ({n_condition/n*100:.1f}%)')
print(f'   🤝 Both models agreed: {n_agree}  ({n_agree/n*100:.1f}%)')
print('\n📊 Conditions found:')
cond_counts = results_df['classification'].value_counts()
max_cnt = cond_counts.max()
for cond, cnt in cond_counts.items():
    bar = '█' * int(cnt / max_cnt * 20)
    print(f'   {cond:<20} : {cnt:>3} ({cnt/n*100:.1f}%)  {bar}')


In [ ]:
print('📊 Chart 1 — Trend Analysis Dashboard')

fig = plt.figure(figsize=(20, 15))
fig.suptitle(
    f'Mental Health Trend Analysis  —  N={n} Participants',
    fontsize=17,
    fontweight='bold',
    y=0.98,
)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)
cond_counts_plot = results_df['classification'].value_counts()


ax1 = fig.add_subplot(gs[0, 0])
pie_colors = [CLASS_COLOR.get(c, '#aaa') for c in cond_counts_plot.index]
wedges, _, autotexts = ax1.pie(
    cond_counts_plot.values,
    labels=cond_counts_plot.index,
    autopct='%1.1f%%',
    colors=pie_colors,
    startangle=140,
    textprops={'fontsize': 8},
    pctdistance=0.80,
)
for at in autotexts:
    at.set_fontsize(7)
ax1.set_title('Condition Distribution', fontweight='bold', fontsize=12)


ax2 = fig.add_subplot(gs[0, 1])
risk_order = ['low', 'borderline', 'high']
risk_labels = {'low': 'Normal', 'borderline': 'Borderline', 'high': 'Detected'}
risk_colors = {'low': '#2ecc71', 'borderline': '#f39c12', 'high': '#e74c3c'}
risk_counts = results_df['risk_level'].value_counts()
present = [r for r in risk_order if r in risk_counts.index]
vals = [risk_counts[r] for r in present]
bars = ax2.bar(
    [risk_labels[r] for r in present],
    vals,
    color=[risk_colors[r] for r in present],
    edgecolor='white',
    width=0.5,
)
for bar, val in zip(bars, vals):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f'{val}\n({val/n*100:.0f}%)',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold',
    )
ax2.set_ylabel('Participants')
ax2.set_ylim(0, max(vals) * 1.35)
ax2.set_title('Mental Health Risk Level', fontweight='bold', fontsize=12)


ax3 = fig.add_subplot(gs[0, 2])
top_conds = cond_counts_plot.head(10)
bars3 = ax3.barh(
    top_conds.index[::-1],
    top_conds.values[::-1],
    color=[CLASS_COLOR.get(c, '#aaa') for c in top_conds.index[::-1]],
    edgecolor='white',
)
for bar, val in zip(bars3, top_conds.values[::-1]):
    ax3.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f'{val}',
        va='center',
        fontsize=9,
        fontweight='bold',
    )
ax3.set_xlabel('Count')
ax3.set_xlim(0, top_conds.max() * 1.3)
ax3.set_title('Top Detected Conditions', fontweight='bold', fontsize=12)


ax4 = fig.add_subplot(gs[1, 0])
conf_vals = results_df['confidence']
ax4.hist(conf_vals, bins=15, color='#3498db', alpha=0.75, edgecolor='white')
ax4.axvspan(0, BORDERLINE_LOWER, alpha=0.10, color='#2ecc71', label='Normal')
ax4.axvspan(
    BORDERLINE_LOWER,
    CONFIDENCE_THRESHOLD,
    alpha=0.10,
    color='#f39c12',
    label='Borderline',
)
ax4.axvspan(CONFIDENCE_THRESHOLD, 1, alpha=0.10, color='#e74c3c', label='Condition')
ax4.axvline(
    conf_vals.mean(),
    color='#e74c3c',
    lw=2,
    linestyle='--',
    label=f'Mean={conf_vals.mean():.2f}',
)
ax4.axvline(
    conf_vals.median(),
    color='#9b59b6',
    lw=2,
    linestyle='--',
    label=f'Median={conf_vals.median():.2f}',
)
ax4.set_xlabel('Confidence Score')
ax4.set_ylabel('Frequency')
ax4.set_title('Confidence Distribution', fontweight='bold', fontsize=12)
ax4.legend(fontsize=7)


ax5 = fig.add_subplot(gs[1, 1:])
conditions_present = results_df['classification'].unique().tolist()
box_data = [
    results_df[results_df['classification'] == c]['confidence'].values
    for c in conditions_present
]
bp = ax5.boxplot(
    box_data, patch_artist=True, medianprops=dict(color='black', linewidth=2)
)
for patch, cname in zip(bp['boxes'], conditions_present):
    patch.set_facecolor(CLASS_COLOR.get(cname, '#aaa'))
    patch.set_alpha(0.75)
ax5.set_xticks(range(1, len(conditions_present) + 1))
ax5.set_xticklabels(conditions_present, rotation=35, ha='right', fontsize=8)
ax5.set_ylabel('Confidence Score')
ax5.set_title('Confidence by Condition', fontweight='bold', fontsize=12)
ax5.axhline(
    CONFIDENCE_THRESHOLD,
    color='#e74c3c',
    lw=1.5,
    linestyle='--',
    label=f'Thr={CONFIDENCE_THRESHOLD}',
)
ax5.axhline(
    BORDERLINE_LOWER,
    color='#f39c12',
    lw=1.5,
    linestyle='--',
    label=f'Border={BORDERLINE_LOWER}',
)
ax5.legend(fontsize=8)


ax6 = fig.add_subplot(gs[2, :])
heat_matrix = blended_matrix_df.values.T
im = ax6.imshow(heat_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=0.6)
risk_icon = {'low': 'N', 'borderline': 'B', 'high': '!'}
for xi, pid in enumerate(blended_matrix_df.index):
    icon = risk_icon[
        results_df.loc[results_df['participant_id'] == pid, 'risk_level'].values[0]
    ]
    ax6.text(xi, -0.9, icon, ha='center', va='top', fontsize=6.5, fontweight='bold')
ax6.set_xticks(range(n))
ax6.set_xticklabels(blended_matrix_df.index, rotation=90, fontsize=6)
ax6.set_yticks(range(len(blended_matrix_df.columns)))
ax6.set_yticklabels(blended_matrix_df.columns, fontsize=8)
ax6.set_title(
    'Blended Probability Heatmap  (N=Normal · B=Borderline · !=Detected)',
    fontweight='bold',
    fontsize=11,
)
plt.colorbar(im, ax=ax6, label='Blended Probability', fraction=0.015)

plt.savefig(
    _paths.output_path('chart1_trend_dashboard.png'), dpi=150, bbox_inches='tight'
)
plt.show()
print('✅ Saved → chart1_trend_dashboard.png')


In [ ]:
import matplotlib.pyplot as plt


risk_order = ['low', 'borderline', 'high']
risk_labels = {'low': 'Normal', 'borderline': 'Borderline', 'high': 'Detected'}
risk_colors = {'low': '#2ecc71', 'borderline': '#f39c12', 'high': '#e74c3c'}
risk_icon = {'low': 'N', 'borderline': 'B', 'high': '!'}
cond_counts_plot = results_df['classification'].value_counts()
conf_vals = results_df['confidence']


fig, ax = plt.subplots(figsize=(7, 6))
pie_colors = [CLASS_COLOR.get(c, '#aaa') for c in cond_counts_plot.index]
wedges, _, autotexts = ax.pie(
    cond_counts_plot.values,
    labels=cond_counts_plot.index,
    autopct='%1.1f%%',
    colors=pie_colors,
    startangle=140,
    textprops={'fontsize': 8},
    pctdistance=0.80,
)
for at in autotexts:
    at.set_fontsize(7)
ax.set_title('Condition Distribution', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(
    _paths.output_path('chart1_condition_distribution.png'),
    dpi=150,
    bbox_inches='tight',
)
plt.show()
print('✅ Saved → chart1_condition_distribution.png')


fig, ax = plt.subplots(figsize=(7, 5))
risk_counts = results_df['risk_level'].value_counts()
present = [r for r in risk_order if r in risk_counts.index]
vals = [risk_counts[r] for r in present]
bars = ax.bar(
    [risk_labels[r] for r in present],
    vals,
    color=[risk_colors[r] for r in present],
    edgecolor='white',
    width=0.5,
)
for bar, val in zip(bars, vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f'{val}\n({val/n*100:.0f}%)',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold',
    )
ax.set_ylabel('Participants')
ax.set_ylim(0, max(vals) * 1.35)
ax.set_title('Mental Health Risk Level', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(_paths.output_path('chart2_risk_level.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → chart2_risk_level.png')


fig, ax = plt.subplots(figsize=(8, 5))
top_conds = cond_counts_plot.head(10)
bars3 = ax.barh(
    top_conds.index[::-1],
    top_conds.values[::-1],
    color=[CLASS_COLOR.get(c, '#aaa') for c in top_conds.index[::-1]],
    edgecolor='white',
)
for bar, val in zip(bars3, top_conds.values[::-1]):
    ax.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f'{val}',
        va='center',
        fontsize=9,
        fontweight='bold',
    )
ax.set_xlabel('Count')
ax.set_xlim(0, top_conds.max() * 1.3)
ax.set_title('Top Detected Conditions', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(
    _paths.output_path('chart3_top_conditions.png'), dpi=150, bbox_inches='tight'
)
plt.show()
print('✅ Saved → chart3_top_conditions.png')


fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(conf_vals, bins=15, color='#3498db', alpha=0.75, edgecolor='white')
ax.axvspan(0, BORDERLINE_LOWER, alpha=0.10, color='#2ecc71', label='Normal')
ax.axvspan(
    BORDERLINE_LOWER,
    CONFIDENCE_THRESHOLD,
    alpha=0.10,
    color='#f39c12',
    label='Borderline',
)
ax.axvspan(CONFIDENCE_THRESHOLD, 1, alpha=0.10, color='#e74c3c', label='Condition')
ax.axvline(
    conf_vals.mean(),
    color='#e74c3c',
    lw=2,
    linestyle='--',
    label=f'Mean={conf_vals.mean():.2f}',
)
ax.axvline(
    conf_vals.median(),
    color='#9b59b6',
    lw=2,
    linestyle='--',
    label=f'Median={conf_vals.median():.2f}',
)
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Frequency')
ax.set_title('Confidence Distribution', fontweight='bold', fontsize=13)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    _paths.output_path('chart4_confidence_distribution.png'),
    dpi=150,
    bbox_inches='tight',
)
plt.show()
print('✅ Saved → chart4_confidence_distribution.png')


fig, ax = plt.subplots(figsize=(10, 5))
conditions_present = results_df['classification'].unique().tolist()
box_data = [
    results_df[results_df['classification'] == c]['confidence'].values
    for c in conditions_present
]
bp = ax.boxplot(
    box_data, patch_artist=True, medianprops=dict(color='black', linewidth=2)
)
for patch, cname in zip(bp['boxes'], conditions_present):
    patch.set_facecolor(CLASS_COLOR.get(cname, '#aaa'))
    patch.set_alpha(0.75)
ax.set_xticks(range(1, len(conditions_present) + 1))
ax.set_xticklabels(conditions_present, rotation=35, ha='right', fontsize=13)
ax.set_ylabel('Confidence Score')
ax.set_title('Confidence by Condition', fontweight='bold', fontsize=14)
ax.axhline(
    CONFIDENCE_THRESHOLD,
    color='#e74c3c',
    lw=1.5,
    linestyle='--',
    label=f'Thr={CONFIDENCE_THRESHOLD}',
)
ax.axhline(
    BORDERLINE_LOWER,
    color='#f39c12',
    lw=1.5,
    linestyle='--',
    label=f'Border={BORDERLINE_LOWER}',
)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(
    _paths.output_path('chart5_confidence_by_condition.png'),
    dpi=150,
    bbox_inches='tight',
)
plt.show()
print('✅ Saved → chart5_confidence_by_condition.png')


fig, ax = plt.subplots(figsize=(max(12, n * 0.35), 6))
heat_matrix = blended_matrix_df.values.T
im = ax.imshow(heat_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=0.6)
for xi, pid in enumerate(blended_matrix_df.index):
    icon = risk_icon[
        results_df.loc[results_df['participant_id'] == pid, 'risk_level'].values[0]
    ]
    ax.text(xi, -0.9, icon, ha='center', va='top', fontsize=6.5, fontweight='bold')
ax.set_xticks(range(n))
ax.set_xticklabels(blended_matrix_df.index, rotation=90, fontsize=6)
ax.set_yticks(range(len(blended_matrix_df.columns)))
ax.set_yticklabels(blended_matrix_df.columns, fontsize=8)
ax.set_title(
    'Blended Probability Heatmap  (N=Normal · B=Borderline · !=Detected)',
    fontweight='bold',
    fontsize=12,
)
plt.colorbar(im, ax=ax, label='Blended Probability', fraction=0.015)
plt.tight_layout()
plt.savefig(_paths.output_path('chart6_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → chart6_heatmap.png')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
agree_by_cond = (
    results_df.groupby('classification')['models_agree']
    .mean()
    .sort_values(ascending=False)
)
bars_a = ax.bar(
    agree_by_cond.index,
    agree_by_cond.values * 100,
    color=[CLASS_COLOR.get(c, '#aaa') for c in agree_by_cond.index],
    edgecolor='white',
)
ax.set_xticklabels(agree_by_cond.index, rotation=40, ha='right', fontsize=10)
ax.set_ylabel('Agreement Rate (%)')
ax.set_ylim(0, 115)
ax.set_title('Model Agreement Rate by Condition', fontweight='bold', fontsize=13)
ax.axhline(50, color='gray', lw=1, linestyle='--')
for bar, val in zip(bars_a, agree_by_cond.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val * 100:.0f}%',
        ha='center',
        va='bottom',
        fontsize=10,
    )
plt.tight_layout()
plt.savefig(
    _paths.output_path('chart3a_agreement_by_condition.png'),
    dpi=150,
    bbox_inches='tight',
)
plt.show()
print('✅ Saved → chart3a_agreement_by_condition.png')


fig, ax = plt.subplots(figsize=(7, 6))
risk_palette = {'low': '#2ecc71', 'borderline': '#f39c12', 'high': '#e74c3c'}
order_r = [
    r for r in ['low', 'borderline', 'high'] if r in results_df['risk_level'].unique()
]
sns.violinplot(
    data=results_df,
    x='risk_level',
    y='confidence',
    order=order_r,
    palette=risk_palette,
    inner='quartile',
    ax=ax,
)
ax.set_xticklabels(['Normal', 'Borderline', 'Detected'][: len(order_r)])
ax.set_xlabel('Risk Level')
ax.set_ylabel('Confidence Score')
ax.set_title('Confidence by Risk Level', fontweight='bold', fontsize=13)
ax.axhline(
    CONFIDENCE_THRESHOLD,
    color='#e74c3c',
    lw=1.5,
    linestyle='--',
    label=f'Thr={CONFIDENCE_THRESHOLD}',
)
ax.axhline(
    BORDERLINE_LOWER,
    color='#f39c12',
    lw=1.5,
    linestyle='--',
    label=f'Border={BORDERLINE_LOWER}',
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    _paths.output_path('chart3b_confidence_by_risk.png'), dpi=150, bbox_inches='tight'
)
plt.show()
print('✅ Saved → chart3b_confidence_by_risk.png')


fig, ax = plt.subplots(figsize=(8, 6))
scatter_colors = [CLASS_COLOR.get(c, '#aaa') for c in results_df['classification']]
ax.scatter(
    results_df['word_count'],
    results_df['confidence'],
    c=scatter_colors,
    alpha=0.75,
    s=60,
    edgecolors='white',
    lw=0.5,
)
ax.axhline(
    CONFIDENCE_THRESHOLD,
    color='#e74c3c',
    lw=1.5,
    linestyle='--',
    label=f'Thr={CONFIDENCE_THRESHOLD}',
)
ax.axhline(
    BORDERLINE_LOWER,
    color='#f39c12',
    lw=1.5,
    linestyle='--',
    label=f'Border={BORDERLINE_LOWER}',
)
ax.set_xlabel('Response Word Count')
ax.set_ylabel('Confidence Score')
ax.set_title('Word Count vs Confidence', fontweight='bold', fontsize=13)
handles_sc = [
    mpatches.Patch(color=CLASS_COLOR.get(c, '#aaa'), label=c)
    for c in results_df['classification'].unique()
]
ax.legend(handles=handles_sc, fontsize=6.5, ncol=2)
plt.tight_layout()
plt.savefig(
    _paths.output_path('chart3c_wordcount_vs_confidence.png'),
    dpi=150,
    bbox_inches='tight',
)
plt.show()
print('✅ Saved → chart3c_wordcount_vs_confidence.png')


In [ ]:
print('📊 Chart 2 — Response Pattern Analysis')


fig1, ax = plt.subplots(figsize=(7, 6))
ax.hist(
    results_df['word_count'], bins=12, color='#9b59b6', alpha=0.75, edgecolor='white'
)
ax.axvline(
    results_df['word_count'].mean(),
    color='#e74c3c',
    lw=2,
    linestyle='--',
    label=f'Mean={results_df["word_count"].mean():.0f} words',
)
ax.set_xlabel('Word Count')
ax.set_ylabel('Participants')
ax.set_title('Response Length Distribution', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(_paths.output_path('chart2a_length_dist.png'), dpi=150, bbox_inches='tight')
plt.show()


fig2, ax = plt.subplots(figsize=(7, 6))
order = (
    results_df.groupby('classification')['word_count']
    .mean()
    .sort_values(ascending=False)
    .index
)
sns.boxplot(
    data=results_df,
    x='classification',
    y='word_count',
    order=order,
    palette={c: CLASS_COLOR.get(c, '#aaa') for c in order},
    ax=ax,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
ax.set_xlabel('')
ax.set_ylabel('Word Count')
ax.set_title('Word Count by Condition', fontweight='bold')
plt.tight_layout()
plt.savefig(
    _paths.output_path('chart2b_wordcount_condition.png'), dpi=150, bbox_inches='tight'
)
plt.show()


fig3, ax = plt.subplots(figsize=(7, 6))
agree_counts = results_df['models_agree'].value_counts()
agree_labels = {True: f'Both agree\n{n_agree}/{n}', False: f'Differ\n{n-n_agree}/{n}'}
wedges, texts, autotexts = ax.pie(
    agree_counts.values,
    labels=[agree_labels.get(k, str(k)) for k in agree_counts.index],
    autopct='%1.0f%%',
    colors=['#2ecc71', '#e74c3c'][: len(agree_counts)],
    startangle=90,
    textprops={'fontsize': 20, 'fontweight': 'bold', 'color': 'black'},
    wedgeprops=dict(linewidth=3, edgecolor='white'),
    pctdistance=0.62,
    labeldistance=1.2,
)
for at in autotexts:
    at.set_fontsize(24)
    at.set_fontweight('bold')
    at.set_color('black')
ax.set_title('Model Agreement\n(ML models vs BART)', fontweight='bold', fontsize=18)
plt.tight_layout()
plt.savefig(
    _paths.output_path('chart2c_model_agreement.png'), dpi=150, bbox_inches='tight'
)
plt.show()

print(
    '✅ Saved → chart2a_length_dist.png, chart2b_wordcount_condition.png, chart2c_model_agreement.png'
)


In [ ]:
print('📊 Chart 3 — Model Agreement & Confidence Deep-Dive')
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Model Agreement & Confidence Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
agree_by_cond = (
    results_df.groupby('classification')['models_agree']
    .mean()
    .sort_values(ascending=False)
)
bars_a = ax.bar(
    agree_by_cond.index,
    agree_by_cond.values * 100,
    color=[CLASS_COLOR.get(c, '#aaa') for c in agree_by_cond.index],
    edgecolor='white',
)
ax.set_xticklabels(agree_by_cond.index, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Agreement Rate (%)')
ax.set_ylim(0, 115)
ax.set_title('Model Agreement Rate by Condition', fontweight='bold')
ax.axhline(50, color='gray', lw=1, linestyle='--')
for bar, val in zip(bars_a, agree_by_cond.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val*100:.0f}%',
        ha='center',
        va='bottom',
        fontsize=8,
    )

ax = axes[1]
risk_palette = {'low': '#2ecc71', 'borderline': '#f39c12', 'high': '#e74c3c'}
order_r = [
    r for r in ['low', 'borderline', 'high'] if r in results_df['risk_level'].unique()
]
sns.violinplot(
    data=results_df,
    x='risk_level',
    y='confidence',
    order=order_r,
    palette=risk_palette,
    inner='quartile',
    ax=ax,
)
ax.set_xticklabels(['Normal', 'Borderline', 'Detected'][: len(order_r)])
ax.set_xlabel('Risk Level')
ax.set_ylabel('Confidence Score')
ax.set_title('Confidence by Risk Level', fontweight='bold')
ax.axhline(CONFIDENCE_THRESHOLD, color='#e74c3c', lw=1.5, linestyle='--')
ax.axhline(BORDERLINE_LOWER, color='#f39c12', lw=1.5, linestyle='--')

ax = axes[2]
scatter_colors = [CLASS_COLOR.get(c, '#aaa') for c in results_df['classification']]
ax.scatter(
    results_df['word_count'],
    results_df['confidence'],
    c=scatter_colors,
    alpha=0.75,
    s=60,
    edgecolors='white',
    lw=0.5,
)
ax.axhline(CONFIDENCE_THRESHOLD, color='#e74c3c', lw=1.5, linestyle='--')
ax.axhline(BORDERLINE_LOWER, color='#f39c12', lw=1.5, linestyle='--')
ax.set_xlabel('Response Word Count')
ax.set_ylabel('Confidence Score')
ax.set_title('Word Count vs Confidence', fontweight='bold')
handles_sc = [
    mpatches.Patch(color=CLASS_COLOR.get(c, '#aaa'), label=c)
    for c in results_df['classification'].unique()
]
ax.legend(handles=handles_sc, fontsize=6.5, ncol=2)

plt.tight_layout()
plt.savefig(
    _paths.output_path('chart3_agreement_confidence.png'), dpi=150, bbox_inches='tight'
)
plt.show()
print('✅ Saved → chart3_agreement_confidence.png')


In [ ]:
print('📊 Chart 4 — Condition Co-occurrence & Model Comparison')
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    'Condition Co-occurrence & Model Comparison', fontsize=14, fontweight='bold'
)

ax = axes[0]
top2_df = results_df[results_df['top2_condition'] != '']
if len(top2_df) > 0:
    co_matrix = pd.crosstab(top2_df['top1_condition'], top2_df['top2_condition'])
    co_matrix = co_matrix.reindex(
        index=[c for c in VALID_CONDITIONS if c in co_matrix.index],
        columns=[c for c in VALID_CONDITIONS if c in co_matrix.columns],
        fill_value=0,
    )
    sns.heatmap(
        co_matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        linewidths=0.5,
        ax=ax,
        cbar_kws={'label': 'Count'},
    )
    ax.set_xlabel('2nd Condition')
    ax.set_ylabel('1st Condition')
    ax.set_title('Condition Co-occurrence\n(Top-1 vs Top-2)', fontweight='bold')

ax = axes[1]
your_counts = results_df['your_model_top'].value_counts()
zs_counts = results_df['zeroshot_top'].value_counts()
all_conds = sorted(set(list(your_counts.index) + list(zs_counts.index)))
x = np.arange(len(all_conds))
w = 0.35
ax.bar(
    x - w / 2,
    [your_counts.get(c, 0) for c in all_conds],
    w,
    label='Ml Models (LR+SVM+RF)',
    color='#3498db',
    alpha=0.85,
    edgecolor='white',
)
ax.bar(
    x + w / 2,
    [zs_counts.get(c, 0) for c in all_conds],
    w,
    label='BART Zero-Shot',
    color='#e74c3c',
    alpha=0.85,
    edgecolor='white',
)
ax.set_xticks(x)
ax.set_xticklabels(all_conds, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Participants')
ax.set_title('ML Models vs BART Top Prediction', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(_paths.output_path('chart4_cooccurrence.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → chart4_cooccurrence.png')


In [ ]:
print('📊 Chart 5 — Condition Probability Profiles')
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Blended Probability Profiles', fontsize=14, fontweight='bold')

ax = axes[0]
mean_probs = blended_matrix_df.mean(axis=0).sort_values(ascending=False)
bars5 = ax.bar(
    mean_probs.index,
    mean_probs.values,
    color=[CLASS_COLOR.get(c, '#aaa') for c in mean_probs.index],
    edgecolor='white',
)
ax.set_xticklabels(mean_probs.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean Blended Probability')
ax.set_title('Average Probability per Condition\n(all participants)', fontweight='bold')
for bar, val in zip(bars5, mean_probs.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f'{val:.3f}',
        ha='center',
        va='bottom',
        fontsize=7,
    )

ax = axes[1]
ax.remove()
ax_r = fig.add_subplot(1, 2, 2, projection='polar')
categories = VALID_CONDITIONS
N = len(categories)
angles = [i / N * 2 * np.pi for i in range(N)] + [0]
mean_vals = [blended_matrix_df[c].mean() for c in categories] + [
    blended_matrix_df[categories[0]].mean()
]
ax_r.set_theta_offset(np.pi / 2)
ax_r.set_theta_direction(-1)
ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels(categories, fontsize=7)
ax_r.plot(angles, mean_vals, 'o-', lw=2, color='#e74c3c')
ax_r.fill(angles, mean_vals, alpha=0.25, color='#e74c3c')
ax_r.set_title('Radar: Group Average\nCondition Profile', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(_paths.output_path('chart5_profiles.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → chart5_profiles.png')


In [ ]:
print('👤 Individual Participant Results')
print('=' * 70)

display_df = results_df[
    [
        'participant_id',
        'classification',
        'confidence',
        'status',
        'top1_condition',
        'top1_prob',
        'top2_condition',
        'top2_prob',
        'models_agree',
        'your_model_top',
        'zeroshot_top',
        'word_count',
    ]
].copy()
display_df['confidence'] = display_df['confidence'].apply(lambda x: f'{x:.1%}')
display_df['top1_prob'] = display_df['top1_prob'].apply(lambda x: f'{x:.1%}')
display_df['top2_prob'] = display_df['top2_prob'].apply(lambda x: f'{x:.1%}')

print(
    f'{"ID":<8} {"Predicted":<20} {"Conf":<8} {"YourModel":<18} {"BART":<18} {"Agree":<7} Status'
)
print('-' * 95)
for _, row in results_df.iterrows():
    icon = (
        'N'
        if row['status'] == 'normal'
        else ('B' if row['status'] == 'borderline' else '!')
    )
    print(
        f"{row['participant_id']:<8} {row['classification']:<20} {row['confidence']:.0%}    "
        f"{row['your_model_top']:<18} {row['zeroshot_top']:<18} {str(row['models_agree']):<7} [{icon}] {row['status']}"
    )

try:
    from IPython.display import display as ipy_display

    def status_color(val):
        return {
            'normal': 'background-color:#d5f5e3',
            'borderline': 'background-color:#fef9e7',
        }.get(val, 'background-color:#fadbd8')

    ipy_display(
        display_df.style.applymap(status_color, subset=['status'])
        .set_properties(**{'font-size': '11px'})
        .set_caption(f'Ensemble Classification — {n} Participants')
    )
except Exception:
    print(display_df.to_string())


In [ ]:
print('💾 Exporting results...')
results_df.to_csv(_paths.output_path('mental_health_results_full.csv'), index=False)
cond_dist = results_df['classification'].value_counts().reset_index()
cond_dist.columns = ['condition', 'count']
cond_dist['percentage'] = (cond_dist['count'] / n * 100).round(1)
cond_dist.to_csv(_paths.output_path('mental_health_distribution.csv'), index=False)
blended_matrix_df.to_csv(_paths.output_path('mental_health_blended_proba.csv'))
summary = {
    'total_participants': [n],
    'normal_count': [n_normal],
    'borderline_count': [n_borderline],
    'condition_detected_count': [n_condition],
    'normal_pct': [f'{n_normal/n*100:.1f}%'],
    'condition_pct': [f'{n_condition/n*100:.1f}%'],
    'mean_confidence': [f'{results_df["confidence"].mean():.1%}'],
    'models_agreed_pct': [f'{n_agree/n*100:.1f}%'],
    'confidence_threshold': [CONFIDENCE_THRESHOLD],
    'borderline_threshold': [BORDERLINE_LOWER],
    'reddit_bart_weights': [f'{REDDIT_WEIGHT:.0%} / {BART_WEIGHT:.0%}'],
    'schema': [' | '.join(VALID_CONDITIONS)],
}
pd.DataFrame(summary).T.reset_index().rename(
    columns={'index': 'metric', 0: 'value'}
).to_csv(_paths.output_path('mental_health_summary.csv'), index=False)
pd.concat(
    [raw_df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1
).to_csv(_paths.output_path('mental_health_merged.csv'), index=False)
print('   ✅ mental_health_results_full.csv')
print('   ✅ mental_health_distribution.csv')
print('   ✅ mental_health_blended_proba.csv')
print('   ✅ mental_health_summary.csv')
print('   ✅ mental_health_merged.csv')
print('\n⚠️  Research Disclaimer: screening tool only — not a clinical diagnosis.')


## 📄 Research Paper — Methods Snippet

> *Text responses were collected via a Google Form survey and analysed using a two-model ensemble. 
A Reddit-trained TF-IDF ensemble (Logistic Regression, Linear SVM, Random Forest) provided 
clinical-vocabulary signal with class-prior bias correction. 
A zero-shot NLI classifier (facebook/bart-large-mnli; Lewis et al., 2020) provided domain-agnostic 
semantic classification using 15 survey-calibrated candidate labels mapped directly to the Reddit 
mental health taxonomy. Final probabilities were determined by a weighted blend (20% TF-IDF, 80% NLI), 
with a keyword-based crisis escalation override for suicidal ideation and a distress-vocabulary-based 
confidence adjustment. Classification thresholds were calibrated to survey language (condition: 0.20, 
borderline: 0.12). SHAP LinearExplainer (Lundberg & Lee, 2017) was applied to the Logistic Regression 
for explainability. All analysis was performed in Python 3.10 using scikit-learn, HuggingFace Transformers, and SHAP.*

**References:**
- Lewis et al. (2020). BART: Denoising Sequence-to-Sequence Pre-training. *ACL 2020*.
- Lundberg & Lee (2017). A unified approach to interpreting model predictions. *NeurIPS 2017*.
- Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR 12*.
